# Phase 3 — Training and Testing a ML Algorithm
## Dataset: GSE6575 — Gene Expression in Blood of Children with Autism Spectrum Disorder

**Algorithm chosen**: Extra Trees Classifier (Extremely Randomized Trees)  
**Goal**: Train and evaluate a classification model to predict the diagnosis group from gene expression profiles.

**Starting point**: We load the prepared data produced in Phase 2 (`X_prepared.csv`, `y_labels.csv`).

In [1]:
# ── Library imports ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, ConfusionMatrixDisplay
)
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
print('All libraries loaded successfully.')

All libraries loaded successfully.


---
## Step 1 — Algorithm Selection and Initial Setup

### 1.1 Algorithm Presentation: Extra Trees Classifier

**Extra Trees** (Extremely Randomized Trees) is an ensemble learning method that builds a large number of unpruned decision trees on the entire training set. It differs from Random Forest in two key ways:

| | Random Forest | Extra Trees |
|---|---|---|
| Split threshold | Best threshold per feature | **Random** threshold per feature |
| Sampling | Bootstrap (subset of samples) | **Full** training set |
| Variance | Lower | Even lower |
| Speed | Moderate | **Faster** |

### 1.2 Justification of Choice

Extra Trees is particularly well-suited for this dataset for the following reasons:

- **High dimensionality (54,630 features, 56 samples)**: Extra Trees handles the curse of dimensionality effectively by randomly selecting split thresholds, which reduces overfitting in high-dimensional spaces.
- **Small sample size**: By using the full training set (no bootstrap), Extra Trees avoids losing precious samples, which is critical when n=56.
- **Robustness to noise**: Gene expression microarray data contains biological and technical noise. The extreme randomization of Extra Trees makes it more robust to noisy features than a single decision tree.
- **Feature importance**: Extra Trees provides feature importance scores, which allow us to identify the most discriminative genes — highly valuable for biological interpretation.
- **Multi-class classification**: The algorithm natively handles the 4-class problem (Autism no regression, Autism with regression, General population, MR·DD).

In [ ]:
# ── Load prepared data from Phase 2 ─────────────────────────────────────────
X = pd.read_csv('X_prepared.csv', index_col=0)
y = pd.read_csv('y_labels.csv', index_col=0).squeeze()

print('Loaded prepared data:')
print(f'  X shape : {X.shape}  (samples × probes)')
print(f'  y shape : {y.shape}  (sample labels)')
print()
print('Class distribution:')
print(y.value_counts().to_string())

### 1.3 Dimensionality Reduction with PCA

With 54,630 features and only 56 samples, training directly is computationally expensive and risks overfitting. We apply **PCA** to reduce dimensionality while retaining 95% of variance.

PCA is applied **after** the train/test split to avoid data leakage (the PCA is fitted only on training data).

In [ ]:
# ── Train/Test Split ─────────────────────────────────────────────────────────
# Stratified split to maintain class proportions in both sets.
# With only 56 samples, we use 80/20 split to maximize training data.

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print('Train/Test split (80/20, stratified):')
print(f'  X_train : {X_train.shape}')
print(f'  X_test  : {X_test.shape}')
print()
print('Class distribution in training set:')
print(y_train.value_counts().to_string())
print()
print('Class distribution in test set:')
print(y_test.value_counts().to_string())

In [ ]:
# ── PCA: fit on training set only, transform both sets ──────────────────────
# n_components=0.95 : keep enough components to explain 95% of variance.
# Fitting on X_train only prevents data leakage into the test set.

pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train)
X_test_pca  = pca.transform(X_test)

print(f'PCA retained {pca.n_components_} components out of {X_train.shape[1]:,} probes')
print(f'Explained variance: {pca.explained_variance_ratio_.sum()*100:.2f}%')
print()
print(f'X_train after PCA: {X_train_pca.shape}')
print(f'X_test  after PCA: {X_test_pca.shape}')

In [ ]:
# ── Visualise cumulative explained variance ──────────────────────────────────

cumvar = np.cumsum(pca.explained_variance_ratio_) * 100

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(cumvar)+1), cumvar, marker='o', markersize=4, color='steelblue')
ax.axhline(95, color='red', linestyle='--', label='95% threshold')
ax.set_xlabel('Number of PCA components')
ax.set_ylabel('Cumulative explained variance (%)')
ax.set_title('PCA — Cumulative Explained Variance')
ax.legend()
plt.tight_layout()
plt.show()

### 1.4 Initial Hyperparameters

We start with the following initial hyperparameters for the Extra Trees Classifier:

| Hyperparameter | Initial Value | Justification |
|---|---|---|
| `n_estimators` | 100 | Standard starting point; enough trees for stable predictions |
| `max_features` | `'sqrt'` | Standard for classification; reduces correlation between trees |
| `min_samples_split` | 2 | Default; allows fine-grained splits given the small sample size |
| `min_samples_leaf` | 1 | Default; allows leaves with a single sample |
| `class_weight` | `'balanced'` | Compensates for class imbalance (MR·DD has only 9 samples) |
| `random_state` | 42 | Ensures reproducibility |
| `n_jobs` | -1 | Uses all available CPU cores for speed |

---
## Step 2 — Model Training

In [ ]:
# ── 2.1 Initial model training ───────────────────────────────────────────────

et_model = ExtraTreesClassifier(
    n_estimators=100,
    max_features='sqrt',
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

et_model.fit(X_train_pca, y_train)

print('Extra Trees model trained successfully.')
print(f'Number of trees : {et_model.n_estimators}')
print(f'Number of PCA features used : {X_train_pca.shape[1]}')

In [ ]:
# ── 2.2 Initial training results ─────────────────────────────────────────────
# We evaluate on the training set to check for overfitting.

y_train_pred = et_model.predict(X_train_pca)
train_accuracy = accuracy_score(y_train, y_train_pred)
train_f1 = f1_score(y_train, y_train_pred, average='weighted')

print('=== Initial Training Results ===')
print(f'Training Accuracy : {train_accuracy:.4f}')
print(f'Training F1-score (weighted) : {train_f1:.4f}')
print()
print('Note: Perfect training accuracy is expected for Extra Trees with default')
print('hyperparameters — the model fits the training data exactly (no pruning).')
print('Generalization performance is evaluated on the test set in Step 3.')

In [ ]:
# ── 2.3 Cross-validation on training set ─────────────────────────────────────
# With only 56 samples, cross-validation gives a more reliable estimate
# of generalization than a single train/test split.
# We use Stratified K-Fold (k=5) to preserve class proportions in each fold.

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Re-run PCA inside cross-validation to avoid leakage
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('pca', PCA(n_components=0.95, random_state=42)),
    ('clf', ExtraTreesClassifier(
        n_estimators=100,
        max_features='sqrt',
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

cv_scores = cross_val_score(pipeline, X, y, cv=skf, scoring='accuracy', n_jobs=-1)
cv_f1     = cross_val_score(pipeline, X, y, cv=skf, scoring='f1_weighted', n_jobs=-1)

print('=== 5-Fold Stratified Cross-Validation Results ===')
print(f'Accuracy per fold : {[f"{s:.3f}" for s in cv_scores]}')
print(f'Mean Accuracy     : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print()
print(f'F1 per fold       : {[f"{s:.3f}" for s in cv_f1]}')
print(f'Mean F1 (weighted): {cv_f1.mean():.4f} ± {cv_f1.std():.4f}')

In [ ]:
# ── 2.4 Challenges encountered and strategies applied ────────────────────────
print('=== Challenges Encountered During Training ===')
print()
print('Challenge 1: High dimensionality (54,630 features vs 56 samples)')
print('  → Strategy: PCA applied before training to reduce to ~N components')
print('               explaining 95% of variance. Prevents memory issues')
print('               and reduces overfitting risk.')
print()
print('Challenge 2: Class imbalance (MR·DD: 9 samples vs Autism w/ regression: 18)')
print('  → Strategy: class_weight="balanced" adjusts class weights inversely')
print('               proportional to class frequencies, preventing bias')
print('               toward majority classes.')
print()
print('Challenge 3: Very small dataset (56 samples total)')
print('  → Strategy: Stratified 5-fold cross-validation instead of single')
print('               train/test split for reliable performance estimation.')

---
## Step 3 — Performance Evaluation

In [ ]:
# ── 3.1 Predictions on the test set ──────────────────────────────────────────

y_pred = et_model.predict(X_test_pca)

test_accuracy = accuracy_score(y_test, y_pred)
test_f1       = f1_score(y_test, y_pred, average='weighted')

print('=== Performance on Test Set ===')
print(f'Test Accuracy         : {test_accuracy:.4f}')
print(f'Test F1-score (weighted): {test_f1:.4f}')
print()
print('Comparison — Training vs Test vs Cross-Validation:')
print(f'  Training Accuracy : {train_accuracy:.4f}')
print(f'  CV Accuracy (mean): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'  Test Accuracy     : {test_accuracy:.4f}')

In [ ]:
# ── 3.2 Detailed classification report ───────────────────────────────────────

print('=== Classification Report (Test Set) ===')
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
# ── 3.3 Confusion matrix ─────────────────────────────────────────────────────

labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(ax=ax, colorbar=True, cmap='Blues')
ax.set_title('Confusion Matrix — Extra Trees (Test Set)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.4 Feature importance via PCA components ────────────────────────────────
# Extra Trees provides feature importances for the PCA components.
# We plot the top 20 most important components.

importances = et_model.feature_importances_
top_n = 20
top_idx = np.argsort(importances)[::-1][:top_n]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(top_n), importances[top_idx], color='steelblue', edgecolor='white')
ax.set_xticks(range(top_n))
ax.set_xticklabels([f'PC{i+1}' for i in top_idx], rotation=45, ha='right')
ax.set_xlabel('PCA Component')
ax.set_ylabel('Feature Importance (Gini)')
ax.set_title('Top 20 Most Important PCA Components — Extra Trees')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.5 Cross-validation score distribution ──────────────────────────────────

fig, ax = plt.subplots(figsize=(7, 4))
folds = [f'Fold {i+1}' for i in range(len(cv_scores))]
ax.bar(folds, cv_scores, color='mediumseagreen', edgecolor='white')
ax.axhline(cv_scores.mean(), color='red', linestyle='--', label=f'Mean = {cv_scores.mean():.3f}')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Accuracy')
ax.set_title('5-Fold Cross-Validation Accuracy — Extra Trees')
ax.legend()
plt.tight_layout()
plt.show()

### 3.6 Analysis of Results

#### Performance metrics interpretation

| Metric | Value | Interpretation |
|---|---|---|
| Training Accuracy | 1.0000 | Expected: Extra Trees fully fits training data (no pruning) |
| CV Accuracy (mean) | *see output* | Reliable generalization estimate across 5 folds |
| Test Accuracy | *see output* | Final unbiased estimate on held-out data |
| F1-score (weighted) | *see output* | Accounts for class imbalance |

#### Comparison to expectations

- **Training accuracy = 1.0** was expected: Extra Trees builds fully grown trees without pruning, so it memorizes training data perfectly. This is normal behavior, not a bug.
- **CV and test accuracy** are the meaningful metrics. Given the very small dataset (56 samples), some variance across folds is expected.
- **Class imbalance** (MR·DD = 9 samples) makes per-class recall vary — the minority class is harder to predict correctly.

#### Strengths of the model

- Handles high-dimensional gene expression data naturally
- Fast training due to random threshold selection
- Provides interpretable feature importances
- `class_weight='balanced'` mitigates the impact of class imbalance
- No assumptions about data distribution (non-parametric)

#### Weaknesses and possible improvements

- **Very small dataset**: 56 samples is insufficient for robust generalization; collecting more samples would significantly improve performance
- **Perfect training accuracy**: indicates potential overfitting despite PCA; increasing `min_samples_leaf` or reducing `n_estimators` could help regularize
- **PCA loses interpretability**: the original gene identities are lost after PCA; alternative feature selection methods (e.g., variance filtering + SelectKBest) could preserve gene-level interpretability
- **Hyperparameter tuning**: a Grid Search or Random Search on `n_estimators`, `max_features`, and `min_samples_leaf` could improve generalization

In [ ]:
# ── 3.7 Final summary ────────────────────────────────────────────────────────
print('=== Phase 3 Summary ===')
print()
print(f'Algorithm          : Extra Trees Classifier')
print(f'Dimensionality     : {X_train.shape[1]:,} probes → {X_train_pca.shape[1]} PCA components (95% variance)')
print(f'Train/Test split   : 80/20 stratified')
print(f'Training samples   : {X_train.shape[0]}')
print(f'Test samples       : {X_test.shape[0]}')
print()
print(f'Training Accuracy  : {train_accuracy:.4f}')
print(f'CV Accuracy (mean) : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Test Accuracy      : {test_accuracy:.4f}')
print(f'Test F1 (weighted) : {test_f1:.4f}')
print()
print('Next step → Phase 4: Training and Testing an Ensemble Method')